# Importaciones

In [1]:
include("dependencies.jl")
include("helpers.jl")
include("wrappers.jl")
# Cargamos los datos preparados en el notebook anterior al instante
JLD2.@load "datos_procesados.jld2" df_trainval df_test X_trainval y_trainval folds_trainval n_features

6-element Vector{Symbol}:
 :df_trainval
 :df_test
 :X_trainval
 :y_trainval
 :folds_trainval
 :n_features

# Modelos básicos y selección de atributos (20%)

In [2]:
# Definición de diccionarios de configuración
dic_filtros = Dict(
    "ANOVA" => MyANOVAFilter(n_features=n_features),
    "Pearson" => MyPearsonFilter(n_features=n_features),
    "Spearman" => MySpearmanFilter(n_features=n_features),
    "Kendall" => MyKendallFilter(n_features=n_features),
    "MI" => MyMIFilter(n_features=n_features),
    "RFE" => MyRFEFilter(n_features=n_features)
)

dic_reducciones = Dict(
    "Sin reducción" => IdentityTransformer(),
    "PCA" => PCA(variance_ratio=0.95),
    "ICA" => ICA(outdim=2, maxiter=10000,tol=0.5),
    "LDA" => LDA(method=:whiten, outdim=5) 
)

dic_modelos = Dict(
    "NeuralNetwork_50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(50,))),
    "NeuralNetwork_100" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100,))),
    "NeuralNetwork_100_50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100, 50))),
    
    "KNN_1" => KNNClassifier(K=1),
    "KNN_10" => KNNClassifier(K=10),
    "KNN_20" => KNNClassifier(K=20),

    "SVM_0.1" => ProbabilisticSVC(cost=0.1),
    "SVM_0.5" => ProbabilisticSVC(cost=0.5),
    "SVM_1.0" => ProbabilisticSVC(cost=1.0)
);

In [3]:
function run_experiment(dic_filtros, dic_reducciones, dic_modelos, output_file; 
                              X=X_trainval, y=y_trainval, folds=folds_trainval)
    
    # --- 1. LÓGICA DE CHECKPOINT ---
    if isfile(output_file)
        println(">>> Archivo de checkpoint encontrado: $output_file")
        
        # Leemos el archivo existente
        results_df = CSV.read(output_file, DataFrame)
        
        # Creamos un conjunto con las combinaciones ya hechas para saltárnoslas
        combinaciones_hechas = Set([
            (r.Filter, r.Reduction, r.Model) 
            for r in eachrow(results_df)
        ])
        
        println(">>> Se han detectado $(length(combinaciones_hechas)) experimentos ya completados. Se saltarán.")
    else
        println(">>> Iniciando experimento desde cero.")
        # Inicializamos el DataFrame vacío
        results_df = DataFrame(
            Filter = String[], Reduction = String[], Model = String[],
            Accuracy = String[],   # Lo ponemos como String para evitar conflictos de tipos
            Accuracy_Mean = Float64[], F1_Score = Float64[], Recall = Float64[]
        )
        combinaciones_hechas = Set{Tuple{String, String, String}}()
    end
    
    # Métricas a evaluar
    measures = [accuracy, multiclass_f1score, recall]

    # --- 2. BUCLE DE EJECUCIÓN ---
    for (filt_name, filt_model) in dic_filtros
        for (red_name, red_model) in dic_reducciones
            for (mod_name, mod_model) in dic_modelos
                
                # Si ya está hecho, saltamos (continue)
                if (filt_name, red_name, mod_name) in combinaciones_hechas
                    continue 
                end

                println(">>> Evaluando: $filt_name + $red_name + $mod_name")
                
                # Instanciamos tu Pipeline Personalizado
                pipe = PersonalizedPipeline(
                    scaler    = MyMinMaxScaler(), 
                    filter    = filt_model,      
                    reduction = red_model,       
                    clf       = mod_model        
                )
                
                try
                    # Evaluación con Cross-Validation
                    evaluation = evaluate!(
                        pipe, X, y,
                        resampling = folds, measures = measures, verbosity = 0,
                        acceleration = CPUThreads()
                    )
                    
                    # Extraer métricas
                    acc_per_fold = evaluation.measurement[1]
                    f1 = evaluation.measurement[2]
                    recall = evaluation.measurement[3]
                    acc_mean = mean(acc_per_fold)
                    
                    println("    Accuracy media: $acc_mean | F1: $f1")
                    
                    # --- CAMBIO DE SEGURIDAD AQUÍ ---
                    # Convertimos el vector a texto y cambiamos la coma por punto y coma
                    # Ejemplo: "[0.9, 0.8]" -> "[0.9; 0.8]"
                    acc_str = replace(string(acc_per_fold), "," => ";")
                    
                    # Guardamos en el DataFrame
                    push!(results_df, (filt_name, red_name, mod_name, acc_str, acc_mean, f1, recall))
                    
                    # Escribimos al archivo CSV inmediatamente
                    CSV.write(output_file, results_df)
                    
                catch e
                    println("!!! Error en $filt_name + $red_name + $mod_name: $e")
                    
                    # Registramos el fallo como "ERROR" en el CSV para no volver a intentarlo en bucle
                    push!(results_df, (filt_name, red_name, mod_name, "ERROR", NaN, NaN, NaN))
                    CSV.write(output_file, results_df)
                end
            end
        end
    end
    
    println("Experimento finalizado.")
    return results_df
end

run_experiment (generic function with 1 method)

In [ ]:
# Ejecutar y guardar
df_resultados_basicos = run_experiment(
    dic_filtros, 
    dic_reducciones, 
    dic_modelos, 
    "resultados_basicos.csv"
)